In [ ]:
# Run annotator.ipynb first to generate amenity category mappings

In [ ]:
import json

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
from tqdm import tqdm

from paths import paths, prepare_preprocessing
from preprocessing.src.constants import CRS_GEO, CRS_METRIC, LSOA_IDS, REGION
from preprocessing.src.geo import get_buffered_region, intersecting_features
from preprocessing.src.isochrone import extract_zones_by_duration
from preprocessing.src.threading import parallelize

In [ ]:
tqdm.pandas()
ox.settings.cache_folder = paths.preprocessing.cache

In [ ]:
prepare_preprocessing()

In [ ]:
AMENITY_OUT_PATH = paths.preprocessing.shared.amenities
GENERATE_MISSING_DATA = not AMENITY_OUT_PATH.exists()

In [ ]:
# Read the amenity categories and filter out benches, as their count dwarfs other leisure amenities
with open(paths.preprocessing.shared.grouped_tags, 'r') as f:
    categories = json.load(f)

categories['leisure'] = [cat for cat in categories['leisure'] if cat[0] != 'amenity' or cat[1] != 'bench']

In [ ]:
# Functions for cleaning the retrieved POI data
def find_matching_tag(row, tag_pairs):
    for key, value in tag_pairs:
        cell = row.get(key)

        if pd.isna(cell):
            continue

        if value in str(cell).split(';'):
            return (key, value)

    return None

def tidy_poi_table(all_poi, categories):
    all_poi = all_poi.copy()
    all_poi['matched_tag'] = all_poi.apply(lambda r: find_matching_tag(r, categories[r['category']]), axis = 1)

    return all_poi[['geometry', 'name', 'category', 'matched_tag']]

In [ ]:
# If the OSM features have not already been fetched, fetch them now, else load from disk
if GENERATE_MISSING_DATA:
    region = get_buffered_region(REGION, 5000, crs = CRS_GEO)

    poi_data = {}

    for category, tag_pairs in tqdm(categories.items(), desc = 'Extracting POI data'):
        tags = {}

        for key, value in tag_pairs:
            tags.setdefault(key, []).append(value)

        poi_data[category] = ox.features_from_polygon(region, tags = tags)

    all_amenities = gpd.GeoDataFrame(
        pd.concat([gdf.assign(category = cat) for cat, gdf in poi_data.items()], ignore_index = True)
    ).to_crs(CRS_METRIC)

    amenities = tidy_poi_table(all_amenities, categories)

    amenities.to_parquet(AMENITY_OUT_PATH, index = False, engine = 'pyarrow')

else:
    amenities = gpd.read_parquet(AMENITY_OUT_PATH)

In [ ]:
# Reuse each seed's larger candidate set while calculating all three walksheds
def process_lsoa(lsoa):
    candidates = {}
    results = {}

    for minutes, zones in extract_zones_by_duration(lsoa, buffer = 15).items():
        zone_counts = []

        for seed, zone in zones:
            subset = intersecting_features(amenities, zone, candidates.get(seed))
            candidates[seed] = subset
            zone_counts.append(subset.groupby('category').size())

        counts_df = pd.DataFrame(zone_counts).fillna(0)
        results[minutes] = (lsoa, counts_df.mean(axis = 0).to_dict())

    return results

# Build the shared spatial index before starting workers (avoid forking native geospatial libraries)
amenities.sindex

results = parallelize(process_lsoa, LSOA_IDS, max_workers = 4, use_threads = True)

In [ ]:
# Shared boundaries are loaded once (each duration receives an independent copy)
lsoa_base = gpd.read_parquet(paths.preprocessing.shared.lsoa)
results_by_duration = {minutes: [result[minutes] for result in results] for minutes in (20, 15, 10)}

In [ ]:
def add_amenity_score(gdf):
    cat_weights = {
        'essentials': 1,
        'eating_out': 1,
        'education': 1,
        'healthcare': 1,
        'culture': 1,
        'leisure': 1,
        'services': 1,
        'vice': 0
    }

    z_cols = []
    active_wts = []

    for cat, w in cat_weights.items():
        raw = f'amenity_count_{cat}_log'
        zcol = f'{raw}_z'

        mean = gdf[raw].mean()
        std = gdf[raw].std(ddof = 0)

        if std == 0 or pd.isna(std):
            gdf[zcol] = 0.0
        else:
            gdf[zcol] = (gdf[raw] - mean) / std

        gdf[zcol] = gdf[zcol] * w

        if w > 0:
            z_cols.append(zcol)
            active_wts.append(w)

    gdf['amenity_score'] = gdf[z_cols].sum(axis = 1) / sum(active_wts)

    return gdf

component_data = {}

for minutes, duration_results in results_by_duration.items():
    lsoa = lsoa_base.copy()
    amenities_per_lsoa_by_type = dict(duration_results)

    for key in categories.keys():
        lsoa[f'amenity_count_{key}'] = (
            lsoa['LSOA21CD'].map(lambda x: amenities_per_lsoa_by_type.get(x).get(key, 0)).fillna(0).astype(float)
        )
        lsoa[f'amenity_count_{key}_log'] = lsoa[f'amenity_count_{key}'].apply(lambda x: np.log1p(x) if x > 0 else 0)

    component_data[minutes] = add_amenity_score(lsoa)

In [ ]:
columns = [
    'LSOA21CD',
    'geometry',
    'amenity_score',
    'amenity_count_essentials',
    'amenity_count_eating_out',
    'amenity_count_education',
    'amenity_count_healthcare',
    'amenity_count_culture',
    'amenity_count_leisure',
    'amenity_count_services',
    'amenity_count_vice'
]

for minutes, lsoa in component_data.items():
    lsoa[columns].to_parquet(paths.preprocessing.walkability[minutes].amenities, index = False)

In [ ]:
for minutes, lsoa in component_data.items():
    FILES = paths.preprocessing.walkability[minutes]

    m = lsoa.explore(
        column = 'amenity_score',
        cmap = 'viridis',
        scheme = 'quantiles',
        k = 50,
        legend = True,
        name = f'{minutes}-min reachable amenities',
        tiles = 'CartoDB positron',
        style_kwds = dict(weight = 0.5, lineOpacity = 0.5, fillOpacity = 0.4),
    )

    m.save(FILES.amenity_map)

m